# CatBoost classifier 
with native categorical feature handling
(no one-hot encoding needed for Gender, City_Type, Current_Car_Type,
Home_Charging_Possible, Subsidy_Available, Range_Anxiety_Level).


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

# Paths / constants

In [2]:
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

sys.path.insert(0, BASE_DIR)

TARGET = "addicted_label"
ID_COL = "id"

TRAIN_PATH = "./dataset/train_fe.csv"
TEST_PATH = "./dataset/test_fe.csv"

MODEL_NAME = "catboost"
ARTIFACT_DIR = "./artifacts"
MODEL_DIR = "./models"
SUB_PATH = f"{ARTIFACT_DIR}/test_pred_{MODEL_NAME}.csv"
OOF_PATH = f"{ARTIFACT_DIR}/oof_{MODEL_NAME}.npy"

FOLD_IDS_PATH = f"{ARTIFACT_DIR}/fold_ids.npy"
N_SPLITS = 5
SEED = 42

NUM_COLS = [
    "age", "daily_screen_time_hours", "social_media_hours", "gaming_hours",
    "work_study_hours", "sleep_hours", "notifications_per_day",
    "app_opens_per_day", "weekend_screen_time",
]
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

CATBOOST_PARAMS = dict(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3.0,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=SEED,
    verbose=200,
    early_stopping_rounds=150,
)

In [3]:
STRESS_MAP = {"Low": 0, "Medium": 1, "High": 2}
YES_NO_MAP = {"Yes": 1, "No": 0}

def load_raw_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    return train, test
 
 
def engineer_features(df):
    df = df.copy()
    df["missing_count"] = df[NUM_COLS + CAT_COLS].isnull().sum(axis=1)
    df["has_any_missing"] = (df["missing_count"] > 0).astype(int)
    df["stress_ord"] = df["stress_level"].map(STRESS_MAP)
    df["academic_impact_bin"] = df["academic_work_impact"].map(YES_NO_MAP)
    df["screen_to_sleep_ratio"] = df["daily_screen_time_hours"] / (df["sleep_hours"] + 1)
    df["social_to_gaming_ratio"] = df["social_media_hours"] / (df["gaming_hours"] + 1)
    df["notifications_per_app_open"] = df["notifications_per_day"] / (df["app_opens_per_day"] + 1)
    df["weekend_vs_weekday_diff"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]
    df["leisure_screen_hours"] = df["daily_screen_time_hours"] - df["work_study_hours"]
    df["sleep_deficit"] = (8 - df["sleep_hours"]).clip(lower=0)
    df["screen_per_notification"] = df["daily_screen_time_hours"] / (df["notifications_per_day"] + 1)
    df["high_stress_flag"] = (df["stress_level"] == "High").astype(float)
    df["low_sleep_flag"] = (df["sleep_hours"] < 6).astype(float)
    df["heavy_screen_flag"] = (
        df["daily_screen_time_hours"] > df["daily_screen_time_hours"].median()
    ).astype(float)
    return df
 
 
def get_feature_lists(df):
    engineered_num = [
        "missing_count", "has_any_missing", "stress_ord", "academic_impact_bin",
        "screen_to_sleep_ratio", "social_to_gaming_ratio", "notifications_per_app_open",
        "weekend_vs_weekday_diff", "leisure_screen_hours", "sleep_deficit",
        "screen_per_notification", "high_stress_flag", "low_sleep_flag", "heavy_screen_flag",
    ]
    num_cols = NUM_COLS + [c for c in engineered_num if c in df.columns]
    cat_cols = [c for c in CAT_COLS if c in df.columns]
    return num_cols, cat_cols

In [4]:
def make_folds(y, n_splits=N_SPLITS, seed=SEED):
    y = np.asarray(y)
    fold_ids = np.full(len(y), -1, dtype=int)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for fold, (_, valid_idx) in enumerate(skf.split(np.zeros(len(y)), y)):
        fold_ids[valid_idx] = fold
    return fold_ids
 
 
def get_or_create_folds(train_df, target_col=TARGET, n_splits=N_SPLITS,
                         seed=SEED, path=FOLD_IDS_PATH):
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    if os.path.exists(path):
        fold_ids = np.load(path)
        if len(fold_ids) == len(train_df):
            return fold_ids
        print("Existing fold file has wrong length, regenerating folds.")
    fold_ids = make_folds(train_df[target_col].values, n_splits=n_splits, seed=seed)
    np.save(path, fold_ids)
    return fold_ids
 
 
def fold_split(train_df, fold_ids, fold):
    train_idx = np.where(fold_ids != fold)[0]
    valid_idx = np.where(fold_ids == fold)[0]
    return train_idx, valid_idx
 
 
def summarize_oof(y_true, oof_pred, model_name="model"):
    auc = roc_auc_score(y_true, oof_pred)
    print(f"[{model_name}] OOF ROC-AUC: {auc:.5f}")
    return auc

In [5]:
def main():
    os.makedirs(ARTIFACT_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR, exist_ok=True)
 
    train, test = load_raw_data()
    train = engineer_features(train)
    test = engineer_features(test)
    num_cols, cat_cols = get_feature_lists(train)
    feature_cols = num_cols + cat_cols
 
    # CatBoost's native cat handling needs string values, not real NaN -
    # fill with an explicit "Missing" placeholder category. Numeric NaNs
    # are left as-is; CatBoost splits on them natively.
    for c in cat_cols:
        train[c] = train[c].fillna("Missing").astype(str)
        test[c] = test[c].fillna("Missing").astype(str)
    cat_feature_idx = [feature_cols.index(c) for c in cat_cols]
 
    y = train[TARGET].values
    fold_ids = get_or_create_folds(train, target_col=TARGET)
 
    oof_pred = np.zeros(len(train))
    test_pred = np.zeros(len(test))
    fold_scores = []
    importances = np.zeros(len(feature_cols))
 
    print("=" * 70)
    print(f"CATBOOST ({N_SPLITS}-fold CV)")
    print("=" * 70)
 
    for fold in range(N_SPLITS):
        train_idx, valid_idx = fold_split(train, fold_ids, fold)
 
        X_train, y_train = train.loc[train_idx, feature_cols], y[train_idx]
        X_valid, y_valid = train.loc[valid_idx, feature_cols], y[valid_idx]
 
        model = CatBoostClassifier(**CATBOOST_PARAMS)
        model.fit(
            X_train, y_train,
            cat_features=cat_feature_idx,
            eval_set=(X_valid, y_valid),
            use_best_model=True,
        )
 
        valid_pred = model.predict_proba(X_valid)[:, 1]
        oof_pred[valid_idx] = valid_pred
 
        fold_auc = roc_auc_score(y_valid, valid_pred)
        fold_scores.append(fold_auc)
        print(f"Fold {fold}: AUC = {fold_auc:.5f} (best_iter={model.get_best_iteration()})")
 
        test_pred += model.predict_proba(test[feature_cols])[:, 1] / N_SPLITS
        importances += np.array(model.get_feature_importance()) / N_SPLITS
        model.save_model(f"{MODEL_DIR}/catboost_fold{fold}.cbm")
 
    print(f"\nMean fold AUC: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})")
    summarize_oof(y, oof_pred, MODEL_NAME)
 
    imp_df = pd.DataFrame({"feature": feature_cols, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False)
    print("\nTop feature importances:")
    print(imp_df.head(15).to_string(index=False))
 
    np.save(OOF_PATH, oof_pred)
    pd.DataFrame({ID_COL: test[ID_COL], TARGET: test_pred}).to_csv(SUB_PATH, index=False)
    print(f"\nSaved OOF predictions -> {OOF_PATH}")
    print(f"Saved test predictions -> {SUB_PATH}")

In [6]:
if __name__ == "__main__":
    main()

CATBOOST (5-fold CV)
0:	test: 0.9105591	best: 0.9105591 (0)	total: 314ms	remaining: 15m 42s
200:	test: 0.9448228	best: 0.9448228 (200)	total: 37.6s	remaining: 8m 44s
400:	test: 0.9520636	best: 0.9520636 (400)	total: 1m 14s	remaining: 8m 1s
600:	test: 0.9558693	best: 0.9558693 (600)	total: 1m 50s	remaining: 7m 20s
800:	test: 0.9579234	best: 0.9579234 (800)	total: 2m 26s	remaining: 6m 42s
1000:	test: 0.9592032	best: 0.9592032 (1000)	total: 3m 5s	remaining: 6m 10s
1200:	test: 0.9600177	best: 0.9600177 (1200)	total: 3m 49s	remaining: 5m 44s
1400:	test: 0.9606958	best: 0.9606958 (1400)	total: 4m 26s	remaining: 5m 4s
1600:	test: 0.9611178	best: 0.9611178 (1600)	total: 5m 4s	remaining: 4m 25s
1800:	test: 0.9615160	best: 0.9615160 (1800)	total: 5m 43s	remaining: 3m 48s
2000:	test: 0.9617971	best: 0.9617971 (2000)	total: 6m 19s	remaining: 3m 9s
2200:	test: 0.9620279	best: 0.9620279 (2200)	total: 6m 56s	remaining: 2m 31s
2400:	test: 0.9622167	best: 0.9622167 (2400)	total: 7m 33s	remaining: 1m 53